# Project 2: Fraud Detection Pipeline (Supervised Learning)

In [10]:
## Step 1: Libraries Import

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_score, recall_score

from imblearn.pipeline import Pipeline 
from imblearn.over_sampling import SMOTE

print("All libraries imported")

All libraries imported


In [11]:
# Step 2: Dataset Load karo

df = pd.read_csv('creditcard.csv')

print("Shape:", df.shape)   # (rows, columns)
df.head()

Shape: (39871, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [12]:
# Step 3: Data Explore

print("Class distribution:")
print(df['Class'].value_counts())

print("\nFraud percentage:", round(df['Class'].mean() * 100, 3), "%")

print("\nMissing values:")
print(df.isnull().sum().sum())  

Class distribution:
Class
0    39767
1      104
Name: count, dtype: int64

Fraud percentage: 0.261 %

Missing values:
0


In [13]:
## Step 4: Separating Features (X) and Target (y)

X = df.drop('Class', axis=1)
y = df['Class']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (39871, 30)
y shape: (39871,)


In [14]:
# Step 5: Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("\nTrain mein fraud %:", round(y_train.mean()*100, 3))
print("Test mein fraud %:", round(y_test.mean()*100, 3))

Train size: (31896, 30)
Test size: (7975, 30)

Train mein fraud %: 0.26
Test mein fraud %: 0.263


In [15]:
# Step 6: Model 1 | Logistic Regression Pipeline

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)
print("Logistic Regression pipeline trained")

Logistic Regression pipeline trained


In [16]:
# Step 7: Model 2 | Random Forest Pipeline

rf_pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_train)
print("Random Forest pipeline trained")

Random Forest pipeline trained


In [17]:
# Step 8: Evaluate Models (Do NOT use Accuracy, use these metrics instead)

def evaluate_model(name, pipeline, X_test, y_test):
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    print(f"===== {name} =====")
    print("Precision:", round(precision_score(y_test, y_pred), 4))
    print("Recall:", round(recall_score(y_test, y_pred), 4))
    print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nFull Report:")
    print(classification_report(y_test, y_pred))
    print("\n")

evaluate_model("Logistic Regression", lr_pipeline, X_test, y_test)
evaluate_model("Random Forest", rf_pipeline, X_test, y_test)

===== Logistic Regression =====
Precision: 0.1353
Recall: 0.8571
ROC-AUC: 0.9386

Confusion Matrix:
[[7839  115]
 [   3   18]]

Full Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      7954
           1       0.14      0.86      0.23        21

    accuracy                           0.99      7975
   macro avg       0.57      0.92      0.61      7975
weighted avg       1.00      0.99      0.99      7975



===== Random Forest =====
Precision: 0.9474
Recall: 0.8571
ROC-AUC: 0.9486

Confusion Matrix:
[[7953    1]
 [   3   18]]

Full Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7954
           1       0.95      0.86      0.90        21

    accuracy                           1.00      7975
   macro avg       0.97      0.93      0.95      7975
weighted avg       1.00      1.00      1.00      7975





In [19]:
# Step 9 (Optional/Advanced): GridSearchCV se Hyperparameter Tuning

param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [10, 20, None],
    'smote__k_neighbors': [3, 5]
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    scoring='roc_auc',   # accuracy nahi, roc_auc pe optimize karenge
    cv=3,                # 3-fold cross validation
    n_jobs=-1,           # sab CPU cores use karo (fast ho jayega)
    verbose=2
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best ROC-AUC (CV):", grid_search.best_score_)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best Parameters: {'classifier__max_depth': 10, 'classifier__n_estimators': 100, 'smote__k_neighbors': 3}
Best ROC-AUC (CV): 0.9912256101897267


In [20]:
# Step 10: Evaluate the Best Model on Final Test Data

best_model = grid_search.best_estimator_
evaluate_model("Best Tuned Random Forest", best_model, X_test, y_test)

===== Best Tuned Random Forest =====
Precision: 0.6667
Recall: 0.8571
ROC-AUC: 0.9953

Confusion Matrix:
[[7945    9]
 [   3   18]]

Full Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7954
           1       0.67      0.86      0.75        21

    accuracy                           1.00      7975
   macro avg       0.83      0.93      0.87      7975
weighted avg       1.00      1.00      1.00      7975





## Conclusion
- Accuracy ko discard karke Precision, Recall, ROC-AUC use kiya
- SMOTE sirf training data par apply hua (leakage-free)
- Do models compare kiye: Logistic Regression vs Random Forest
- GridSearchCV se best hyperparameters dhunde, wo bhi safely (leakage ke bina)

Agla step: Jis model ka Recall aur ROC-AUC best ho, usi ko final model
ke taur pe select karo aur apne report/presentation mein results daalo.